# Ground-Truth Comparison Against Inspec

This notebook reproduces the external ground-truth comparison reported in the
manuscript (Section 3.1.2, "Comparison Against an External Ground-Truth
Benchmark"). The same extraction pipeline used for the mathematics-education
corpus (LLaMA 3.1 8B-instruct, identical prompt and decoding configuration,
see `1. LLMS.ipynb`) is applied to the Inspec keyphrase-extraction benchmark
(Hulth, 2003), and the generated keywords are compared against Inspec's
gold-standard keyphrases.

Inputs:
- `dataset_inspec.csv`: 2,000 Inspec abstracts with gold keyphrases
  (`keywords_gt`) and the exact text (`insumo`, title + abstract only, no
  separate keyword field) that was fed to the LLM.
- `inspec_llama-3.1-8b-EN.csv`: the LLM's generated keywords for each of the
  2,000 abstracts (`keywords_llm`), in the same row order as `dataset_inspec.csv`.

All similarity computations use the same multilingual embedding model used
throughout this study (`paraphrase-multilingual-MiniLM-L12-v2`), so results
are directly comparable to Table 2 of the manuscript.


In [ ]:
pip install sentence-transformers


In [2]:
import ast, os, re
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

PATH_INSPEC_GT = "dataset_inspec.csv"
PATH_LLAMA_8B  = "inspec_llama-3.1-8b-EN.csv"


@dataclass
class Config:
    tau: float = 0.70  # semantic soft-matching threshold
    model_name: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    batch_size: int = 128
    keep_null: bool = False


CFG = Config()


In [3]:
_whitespace_re = re.compile(r"\s+")
_non_alnum_space_re = re.compile(r"[^a-z0-9 ]+")


def normalize_phrase(s):
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = _whitespace_re.sub(" ", s)
    s = _non_alnum_space_re.sub(" ", s)
    return _whitespace_re.sub(" ", s).strip()


def safe_parse_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(i) for i in x]
    s = str(x).strip()
    if not s:
        return []
    try:
        v = ast.literal_eval(s)
        return [str(i) for i in v] if isinstance(v, list) else []
    except Exception:
        s = s.strip("[]")
        return [p.strip().strip('"').strip("'") for p in s.split(",") if p.strip()]


def clean_list(items, keep_null=False):
    out = []
    for it in items:
        t = normalize_phrase(it)
        if not t or (t == "null" and not keep_null):
            continue
        out.append(t)
    seen, dedup = set(), []
    for t in out:
        if t not in seen:
            seen.add(t)
            dedup.append(t)
    return dedup


In [4]:
def jaccard(a, b):
    A, B = set(a), set(b)
    if not A and not B:
        return 1.0
    if not A or not B:
        return 0.0
    return len(A & B) / len(A | B)


def cosine_sim_matrix(X, Y):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    Yn = Y / (np.linalg.norm(Y, axis=1, keepdims=True) + 1e-12)
    return Xn @ Yn.T


def soft_matching_metrics(gt_phrases, llm_phrases, embed_fn, tau):
    """Soft precision/recall/F1 (coverage-style) plus soft mean-max."""
    if not gt_phrases and not llm_phrases:
        return {"soft_recall": 1.0, "soft_precision": 1.0, "soft_f1": 1.0, "soft_mean_max": 1.0}
    if not gt_phrases or not llm_phrases:
        return {"soft_recall": 0.0, "soft_precision": 0.0, "soft_f1": 0.0, "soft_mean_max": 0.0}

    E_gt = embed_fn(gt_phrases)
    E_llm = embed_fn(llm_phrases)
    S = cosine_sim_matrix(E_gt, E_llm)

    gt_best = S.max(axis=1)
    llm_best = S.max(axis=0)

    soft_recall = float(np.mean(gt_best >= tau))
    soft_precision = float(np.mean(llm_best >= tau))
    soft_f1 = 0.0 if (soft_recall + soft_precision) == 0 else float(
        2 * soft_recall * soft_precision / (soft_recall + soft_precision)
    )
    soft_mean_max = float(0.5 * (gt_best.mean() + llm_best.mean()))

    return {
        "soft_recall": soft_recall,
        "soft_precision": soft_precision,
        "soft_f1": soft_f1,
        "soft_mean_max": soft_mean_max,
    }


def global_concat_similarity(gt_phrases, llm_phrases, embed_fn):
    """Cosine similarity between the two keyphrase sets, each concatenated with " ; "."""
    if not gt_phrases and not llm_phrases:
        return 1.0
    if not gt_phrases or not llm_phrases:
        return 0.0
    gt_text = " ; ".join(gt_phrases)
    llm_text = " ; ".join(llm_phrases)
    E = embed_fn([gt_text, llm_text])
    x, y = E[0], E[1]
    return float(np.dot(x, y) / ((np.linalg.norm(x) * np.linalg.norm(y)) + 1e-12))


In [5]:
print(f"Loading embedding model: {CFG.model_name}")
model = SentenceTransformer(CFG.model_name)


def embed_fn(texts):
    return model.encode(
        texts,
        batch_size=CFG.batch_size,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=False,
    )


gt_df = pd.read_csv(PATH_INSPEC_GT)
gt_df["row_abs"] = np.arange(len(gt_df))
l8 = pd.read_csv(PATH_LLAMA_8B)

df = gt_df.merge(l8[["row_abs", "keywords_llm"]], on="row_abs", how="left")
df = df.rename(columns={"keywords_llm": "keywords_llm_8b"})
print(f"Rows: {len(df)}  |  Missing 8B outputs: {df['keywords_llm_8b'].isna().sum()}")

records = []
for _, row in df.iterrows():
    gt_list = clean_list(safe_parse_list(row.get("keywords_gt")), keep_null=False)
    llm_list = clean_list(safe_parse_list(row.get("keywords_llm_8b")), keep_null=CFG.keep_null)

    jac = jaccard(gt_list, llm_list)
    soft = soft_matching_metrics(gt_list, llm_list, embed_fn, CFG.tau)
    glob = global_concat_similarity(gt_list, llm_list, embed_fn)

    records.append({
        "row_abs": int(row["row_abs"]),
        "gt_n": len(gt_list),
        "llm_n": len(llm_list),
        "jaccard_lex": jac,
        "soft_recall": soft["soft_recall"],
        "soft_precision": soft["soft_precision"],
        "soft_f1": soft["soft_f1"],
        "soft_mean_max": soft["soft_mean_max"],
        "global_sem_sim": glob,
    })

per_doc = pd.DataFrame(records)


Loading embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Rows: 2000  |  Missing 8B outputs: 0


In [6]:
metrics_cols = [
    "jaccard_lex", "soft_recall", "soft_precision", "soft_f1", "soft_mean_max", "global_sem_sim",
]
print("=== SUMMARY (LLaMA 3.1 8B, multilingual embedder, tau=0.70) ===")
for c in metrics_cols:
    s = per_doc[c]
    print(f"{c:16s} mean={s.mean():.4f}  sd={s.std(ddof=1):.4f}")

# Expected values (as reported in the manuscript, Section 3.1.2):
# jaccard_lex      mean=0.1422
# soft_recall      mean=0.4670
# soft_precision   mean=0.7939
# soft_f1          mean=0.5652
# soft_mean_max    mean=0.7522
# global_sem_sim   mean=0.8042

per_doc.to_csv("inspec_eval_8b_per_document.csv", index=False)
print("\nSaved: inspec_eval_8b_per_document.csv")


=== SUMMARY (LLaMA 3.1 8B, multilingual embedder, tau=0.70) ===
jaccard_lex      mean=0.1422  sd=0.1118
soft_recall      mean=0.4670  sd=0.1799
soft_precision   mean=0.7939  sd=0.2069
soft_f1          mean=0.5652  sd=0.1626
soft_mean_max    mean=0.7522  sd=0.0791
global_sem_sim   mean=0.8042  sd=0.0785

Saved: inspec_eval_8b_per_document.csv
